# Next-Gen Malayalam Tokenizers Seed-17 DGX Batch
Evaluates the 3 next-gen 48K arms (`T8-LS-9999`, `M48-Guard-9999`, `ML-MinGram-WP-G1000`) under both `fixed_source_bytes` (50 MB) and `fixed_training_tokens` (5.0M tokens), then computes the paired 7-arm comparison (`M48`, `F-space`, `T5`, `T8`, `T8-LS-9999`, `M48-Guard-9999`, `ML-MinGram-WP-G1000`) with 10,000-draw document bootstrap.
Pin the GitHub deployment commit with `MOM_BPE_DEPLOY_REVISION`. All safety toggles default to `False`, and Xet / hf_transfer are forced off (`HF_HUB_DISABLE_XET="1"`, `HF_HUB_ENABLE_HF_TRANSFER="0"`).


## 1. Verify the pinned Python environment


In [ ]:
import os
import subprocess

python = "/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python"
probe = """
import importlib.metadata as md
import platform
import unicodedata
import torch

versions = {"regex": "2026.9.29", "tokenizers": "0.23.3", "PyYAML": "6.0.3", "fasttext-wheel": "0.9.2",
            "pyarrow": "26.0.0", "numpy": "2.5.2", "torch": "2.5.1+cu121"}
observed = {name: md.version(name) for name in versions}
runtime = (platform.system(), platform.python_version(), unicodedata.unidata_version, torch.version.cuda)
if runtime != ("Linux", "3.12.0", "15.0.0", "12.1") or observed != versions:
    raise RuntimeError(f"DGX environment differs: {runtime}, {observed}")
md.version("huggingface_hub")
print("Verified the pinned DGX Python environment and Hugging Face client")
"""
env = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="-1",
    HF_HUB_OFFLINE="1",
    HF_DATASETS_OFFLINE="1",
    HF_HUB_DISABLE_XET="1",
    HF_HUB_ENABLE_HF_TRANSFER="0",
)
subprocess.run([python, "-I", "-B", "-c", probe], env=env, check=True)


## 2. Retrieve exact GitHub source commits


In [ ]:
import fcntl
import os
import re
import subprocess
from pathlib import Path
from tempfile import mkdtemp

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

repository = "https://github.com/Akshai0007/Thesi_Tokenizer_Pipeline.git"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("HF_TOKEN", None)
env.pop("HUGGING_FACE_HUB_TOKEN", None)

def git(directory, *args):
    return subprocess.check_output(["git", "-C", str(directory), *args], env=env, text=True).strip()

with (root / ".nextgen-source.lock").open("a+b") as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    for directory, pin in ((deployment, revision), (source, operational)):
        if not directory.exists():
            staging = Path(mkdtemp(prefix="mom-source-", dir=root))
            git(staging, "init", "--quiet")
            git(staging, "fetch", "--no-tags", "--depth=1", repository, pin)
            git(staging, "checkout", "--detach", "--quiet", "FETCH_HEAD")
            if git(staging, "rev-parse", "HEAD") != pin or directory.exists():
                raise RuntimeError("GitHub checkout revision or destination conflict")
            staging.rename(directory)
        if git(directory, "rev-parse", "HEAD") != pin:
            raise RuntimeError(f"GitHub commit mismatch at {directory.name}")
print("Retrieved both pinned GitHub checkouts")


## 3. Verify operational parity, deployment hashes, and environment


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES="-1",
        HF_HUB_DISABLE_TELEMETRY="1",
        HF_HUB_DISABLE_XET="1",
        HF_HUB_ENABLE_HF_TRANSFER="0",
        HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "analyze", "backup-hub") else "1",
        HF_DATASETS_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("nextgen_dgx.py", "nextgen_pack.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run(
        [str(python), "-I", "-B", str(directory / "nextgen_dgx.py"), action,
         "--root", str(root), "--source", str(source), "--deployment-revision", revision,
         "--session", session, *options],
        cwd=source, env=env, check=True,
    )

run("source")
run("environment")


## 4. Authenticate to private Hugging Face repositories (read-only)


In [ ]:
import os
import re
import subprocess
from getpass import getpass
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

inputs_repo = "Akzhai/mom-bpe-nextgen-inputs-private"
inputs_revision = os.environ.get("MOM_BPE_NEXTGEN_INPUTS_REVISION", "main")
results_repo = "Akzhai/mom-bpe-dgx-results-private"

token = getpass("Hugging Face read-only token (blank uses approved cached credentials): ")
prior_token = os.environ.get("HF_TOKEN")
try:
    if token:
        os.environ["HF_TOKEN"] = token
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES="-1",
        HF_HUB_DISABLE_TELEMETRY="1",
        HF_HUB_DISABLE_XET="1",
        HF_HUB_ENABLE_HF_TRANSFER="0",
        HF_HUB_OFFLINE="0",
        HF_DATASETS_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    env.pop("MLTOK_PLUGINS", None)
    subprocess.run(
        [str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), "authenticate",
         "--root", str(root), "--source", str(source),
         "--inputs-repo", inputs_repo, "--inputs-revision", inputs_revision, "--results-repo", results_repo],
        cwd=source, env=env, check=True,
    )
finally:
    if prior_token is None:
        os.environ.pop("HF_TOKEN", None)
    else:
        os.environ["HF_TOKEN"] = prior_token
    del token


## 5. Restore next-gen inputs & Seed-17 baselines individually


In [ ]:
import os
import re
import subprocess
from getpass import getpass
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

inputs_repo = "Akzhai/mom-bpe-nextgen-inputs-private"
inputs_revision = os.environ.get("MOM_BPE_NEXTGEN_INPUTS_REVISION", "main")
results_repo = "Akzhai/mom-bpe-dgx-results-private"
restore_baselines = True

token = getpass("Hugging Face read-only token (blank if local inputs or cached token exist): ")
prior_token = os.environ.get("HF_TOKEN")
try:
    if token:
        os.environ["HF_TOKEN"] = token
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES="-1",
        HF_HUB_DISABLE_TELEMETRY="1",
        HF_HUB_DISABLE_XET="1",
        HF_HUB_ENABLE_HF_TRANSFER="0",
        HF_HUB_OFFLINE="0",
        HF_DATASETS_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    env.pop("MLTOK_PLUGINS", None)
    cmd = [
        str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), "restore-individual",
        "--root", str(root), "--source", str(source),
        "--inputs-repo", inputs_repo, "--inputs-revision", inputs_revision, "--results-repo", results_repo,
    ]
    if restore_baselines:
        cmd.append("--restore-baselines")
    subprocess.run(cmd, cwd=source, env=env, check=True)
finally:
    if prior_token is None:
        os.environ.pop("HF_TOKEN", None)
    else:
        os.environ["HF_TOKEN"] = prior_token
    del token


## 6. Verify all next-gen caches, model hashes, 6 exposure plans, and scratch capacity


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

def run(action, *options):
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES="-1",
        HF_HUB_DISABLE_TELEMETRY="1",
        HF_HUB_DISABLE_XET="1",
        HF_HUB_ENABLE_HF_TRANSFER="0",
        HF_HUB_OFFLINE="1",
        HF_DATASETS_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    env.pop("MLTOK_PLUGINS", None)
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run(
        [str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), action,
         "--root", str(root), "--source", str(source), "--deployment-revision", revision,
         "--session", session, *options],
        cwd=source, env=env, check=True,
    )

run("verify")
run("plan")
run("ready", "--scope", "all")


## 7. Inspect GPUs and batch scratch space


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

env = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="-1",
    HF_HUB_DISABLE_TELEMETRY="1",
    HF_HUB_DISABLE_XET="1",
    HF_HUB_ENABLE_HF_TRANSFER="0",
    HF_HUB_OFFLINE="1",
    HF_DATASETS_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
)
env.pop("MLTOK_PLUGINS", None)
session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
subprocess.run(
    [str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), "inspect",
     "--root", str(root), "--source", str(source), "--deployment-revision", revision, "--session", session],
    cwd=source, env=env, check=True,
)


## 8. Run approved A100 admission & smoke test


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

smoke_approved = False
allocated = False
shared_approved = False
uuid = ""
scope = "all"

if smoke_approved and allocated:
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES="-1",
        HF_HUB_DISABLE_TELEMETRY="1",
        HF_HUB_DISABLE_XET="1",
        HF_HUB_ENABLE_HF_TRANSFER="0",
        HF_HUB_OFFLINE="1",
        HF_DATASETS_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    env.pop("MLTOK_PLUGINS", None)
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    cmd = [
        str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), "smoke",
        "--root", str(root), "--source", str(source), "--deployment-revision", revision,
        "--session", session, "--enable", "--allocated", "--uuid", uuid, "--scope", scope,
    ]
    if shared_approved:
        cmd.append("--shared")
    subprocess.run(cmd, cwd=source, env=env, check=True)
else:
    print("A100 diagnostic disabled")


## 9. Execute Pair 1 (`fixed_source_bytes`: 3 arms) & verify gate


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

enable_training = False

if enable_training:
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES="-1",
        HF_HUB_DISABLE_TELEMETRY="1",
        HF_HUB_DISABLE_XET="1",
        HF_HUB_ENABLE_HF_TRANSFER="0",
        HF_HUB_OFFLINE="1",
        HF_DATASETS_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    env.pop("MLTOK_PLUGINS", None)
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    script = str(deployment / "experiments/mom_bpe/nextgen_dgx.py")
    base = [str(python), "-I", "-B", script, "first", "--root", str(root), "--source", str(source),
            "--deployment-revision", revision, "--session", session, "--enable"]
    subprocess.run(base, cwd=source, env=env, check=True)
    gate = [str(python), "-I", "-B", script, "results", "--root", str(root), "--source", str(source),
            "--deployment-revision", revision, "--session", session, "--scope", "first"]
    subprocess.run(gate, cwd=source, env=env, check=True)
else:
    print("GPU training disabled (fixed_source_bytes)")


## 10. Execute Pair 2 (`fixed_training_tokens`: 3 arms) & verify all 6 results


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

enable_training = False

if enable_training:
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES="-1",
        HF_HUB_DISABLE_TELEMETRY="1",
        HF_HUB_DISABLE_XET="1",
        HF_HUB_ENABLE_HF_TRANSFER="0",
        HF_HUB_OFFLINE="1",
        HF_DATASETS_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    env.pop("MLTOK_PLUGINS", None)
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    script = str(deployment / "experiments/mom_bpe/nextgen_dgx.py")
    subprocess.run(
        [str(python), "-I", "-B", script, "second", "--root", str(root), "--source", str(source),
         "--deployment-revision", revision, "--session", session, "--enable"],
        cwd=source, env=env, check=True,
    )
    subprocess.run(
        [str(python), "-I", "-B", script, "results", "--root", str(root), "--source", str(source),
         "--deployment-revision", revision, "--session", session, "--scope", "all"],
        cwd=source, env=env, check=True,
    )
else:
    print("GPU training disabled (fixed_training_tokens)")


## 11. Generate 7-arm paired development comparisons & 10,000-draw bootstrap


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

env = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="-1",
    HF_HUB_DISABLE_TELEMETRY="1",
    HF_HUB_DISABLE_XET="1",
    HF_HUB_ENABLE_HF_TRANSFER="0",
    HF_HUB_OFFLINE="1",
    HF_DATASETS_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
)
env.pop("MLTOK_PLUGINS", None)
session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
subprocess.run(
    [str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), "analyze",
     "--root", str(root), "--source", str(source), "--deployment-revision", revision, "--session", session],
    cwd=source, env=env, check=True,
)


## 12. Optionally back up verified next-gen outputs to private Hugging Face (`seed17/nextgen/...`)


In [ ]:
import os
import re
import subprocess
from getpass import getpass
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

backup_approved = False
checkpoint_backup_approved = False
repository = "Akzhai/mom-bpe-dgx-results-private"

if backup_approved:
    write_token = getpass("Hugging Face write-scoped token for private backup (cleaned up immediately): ")
    prior_token = os.environ.get("HF_TOKEN")
    try:
        if write_token:
            os.environ["HF_TOKEN"] = write_token
        env = dict(
            os.environ,
            CUDA_VISIBLE_DEVICES="-1",
            HF_HUB_DISABLE_TELEMETRY="1",
            HF_HUB_DISABLE_XET="1",
            HF_HUB_ENABLE_HF_TRANSFER="0",
            HF_HUB_OFFLINE="0",
            HF_DATASETS_OFFLINE="1",
            TRANSFORMERS_OFFLINE="1",
        )
        env.pop("MLTOK_PLUGINS", None)
        session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
        cmd = [
            str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), "backup-hub",
            "--root", str(root), "--source", str(source), "--deployment-revision", revision,
            "--session", session, "--enable", "--hub-repo", repository, "--scope", "all",
        ]
        if checkpoint_backup_approved:
            cmd.append("--hub-checkpoints")
        subprocess.run(cmd, cwd=source, env=env, check=True)
    finally:
        if prior_token is None:
            os.environ.pop("HF_TOKEN", None)
        else:
            os.environ["HF_TOKEN"] = prior_token
        del write_token
else:
    print("Private backup disabled")


## 13. Write final session report


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)

env = dict(
    os.environ,
    CUDA_VISIBLE_DEVICES="-1",
    HF_HUB_DISABLE_TELEMETRY="1",
    HF_HUB_DISABLE_XET="1",
    HF_HUB_ENABLE_HF_TRANSFER="0",
    HF_HUB_OFFLINE="1",
    HF_DATASETS_OFFLINE="1",
    TRANSFORMERS_OFFLINE="1",
)
env.pop("MLTOK_PLUGINS", None)
session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
subprocess.run(
    [str(python), "-I", "-B", str(deployment / "experiments/mom_bpe/nextgen_dgx.py"), "report",
     "--root", str(root), "--source", str(source), "--deployment-revision", revision, "--session", session],
    cwd=source, env=env, check=True,
)
